# Implicit Functions and Automatic Differentiation

Let $z^\star$ solve

$$
F(z^\star,\theta)=f_\theta(z^\star,x)-z^\star=0.
$$

Differentiate $F=0$:

$$
\frac{\partial f_\theta}{\partial z}\,dz
+\frac{\partial f_\theta}{\partial \theta}\,d\theta
-dz=0.
$$

Collect the $dz$ terms:

$$
(I-J_f)\,dz=\frac{\partial f_\theta}{\partial \theta}\,d\theta.
$$

Reverse-mode differentiation solves the transposed linear system

$$
(I-J_f^\top)u=g,
$$

where $g=\partial \ell/\partial z^\star$.

<!-- silva-numbered-citations:start -->
**Numbered literature:** [3](https://jseluis.github.io/silva-networks/paper/references/#ref-3), [4](https://jseluis.github.io/silva-networks/paper/references/#ref-4), [13](https://jseluis.github.io/silva-networks/paper/references/#ref-13). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

from silva_networks import resolve_device, SolverConfig

torch.manual_seed(7)
np.random.seed(7)
device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
device

## Materialized Jacobian on a Small State

For a tiny state it is useful to materialize $J_f$. This makes the
matrix-free `vjp` and `jvp` calls easy to verify.

In [ ]:
from silva_networks import (
    SILVAImplicitTransition,
    fixed_point,
    full_jacobian,
    vjp,
    jvp,
    implicit_adjoint_solve,
)

transition = SILVAImplicitTransition(2, 2, spectral_scale=0.35).to(device)
x = torch.tensor([[0.3, -0.5]], device=device)
z0 = torch.zeros(1, 2, device=device)

def f(z):
    return transition(z, x)

solve = fixed_point(f, z0, SolverConfig(max_iter=20, alpha=0.7))
J = full_jacobian(f, solve.z)
J

In [ ]:
probe = torch.randn_like(solve.z)
_, jvp_value = jvp(f, solve.z, probe)
vjp_value = vjp(f, solve.z, probe)

checks = {
    "Jv_close": torch.allclose(J @ probe.reshape(-1), jvp_value.reshape(-1), atol=1e-5),
    "JT_v_close": torch.allclose(J.T @ probe.reshape(-1), vjp_value.reshape(-1), atol=1e-5),
}
checks

## Adjoint Solve

The explicit matrix solution is

$$
u=(I-J_f^\top)^{-1}g.
$$

The package helper computes the same object with VJP-backed GMRES.

In [ ]:
g = torch.tensor([[1.0, -0.25]], device=device)
explicit_u = torch.linalg.solve(torch.eye(2, device=device) - J.T, g.reshape(-1)).reshape_as(g)
gmres_u = implicit_adjoint_solve(f, solve.z, g, max_iter=8, tol=1e-7)

explicit_u.detach().cpu().numpy(), gmres_u.x.detach().cpu().numpy(), gmres_u.residuals

## Jacobian Spectrum

The spectral radius $\rho(J_f)$ is a local fixed-point diagnostic. A value
below one is compatible with local contraction. A value near or above one means
the solver may need damping, a different solver, or regularization.

In [ ]:
from silva_networks import spectral_radius, hutchinson_jacobian_norm

rho = spectral_radius(f, solve.z, iters=12)
fro = hutchinson_jacobian_norm(f, solve.z, samples=2, squared=False)
float(rho), float(fro.detach().cpu())

## Citation and Sources

If this package or notebook is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.2.0. MIT License.
https://github.com/jseluis/silva-networks
https://doi.org/10.5281/zenodo.21770098
```

When the work is connected to the SILVA methodology, cite the SILVA Networks
paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background sources:

- Deep Implicit Layers tutorial: https://implicit-layers-tutorial.org/
- LocusLab DEQ repository: https://github.com/locuslab/deq
- Deep Equilibrium Models: https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models: https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization: https://arxiv.org/abs/2106.14342

The notebook is adapted to the `silva_networks` public API. It links to the
sources above and keep the examples package-native.

## From 02 Implicit Autodiff to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | the converged latent state z_star |
| Condition | differentiable parameters and external inputs |
| Repeated computation | the map whose Jacobian defines I - J_z f |
| Required invariants | agreement of dense, JVP, VJP, and matrix-free products |
| Replaceable components | transition, Jacobian product, linear solver, regularizer, and loss |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


In [ ]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**adjoint residual and gradient error against explicit differentiation**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **state dimension and matrix-free linear-solver iterations**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [ ]:
notebook_reproduction_record = {
    "notebook": '02_implicit_autodiff.ipynb',
    "state": 'the converged latent state z_star',
    "condition": 'differentiable parameters and external inputs',
    "transition": 'the map whose Jacobian defines I - J_z f',
    "invariants": 'agreement of dense, JVP, VJP, and matrix-free products',
    "compact_metric": 'adjoint residual and gradient error against explicit differentiation',
    "scale_axis": 'state dimension and matrix-free linear-solver iterations',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record


## Where to Go Next

| Question | Page |
| --- | --- |
| How is the backward linear system implemented? | [Implicit Backward Guide](https://jseluis.github.io/silva-networks/learn/implicit-backward-guide/) |
| Where is implicit differentiation derived? | [Mathematical Foundations](https://jseluis.github.io/silva-networks/learn/mathematical-foundations/#implicit-differentiation) |
| Which engine controls expose backward solving? | [DEQ Engine API](https://jseluis.github.io/silva-networks/api/deq-engine/) |
